In [0]:
# Load all insurance datasets

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([(1, 'Cust1', 25, 'Bangalore'), (2, 'Cust2', 52, 'Chennai'), (3, 'Cust3', 46, 'Bangalore'), (4, 'Cust4', 58, 'Hyderabad'), (5, 'Cust5', 36, 'Hyderabad'), (6, 'Cust6', 50, 'Mumbai'), (7, 'Cust7', 43, 'Chennai'), (8, 'Cust8', 32, 'Chennai'), (9, 'Cust9', 47, 'Hyderabad'), (10, 'Cust10', 39, 'Mumbai'), (11, 'Cust11', 42, 'Hyderabad'), (12, 'Cust12', 33, 'Bangalore'), (13, 'Cust13', 57, 'Hyderabad'), (14, 'Cust14', 60, 'Mumbai'), (15, 'Cust15', 30, 'Hyderabad'), (16, 'Cust16', 56, 'Bangalore'), (17, 'Cust17', 51, 'Chennai'), (18, 'Cust18', 36, 'Mumbai'), (19, 'Cust19', 35, 'Hyderabad'), (20, 'Cust20', 43, 'Bangalore'), (21, 'Cust21', 45, 'Chennai'), (22, 'Cust22', 40, 'Hyderabad'), (23, 'Cust23', 30, 'Mumbai'), (24, 'Cust24', 24, 'Mumbai'), (25, 'Cust25', 49, 'Bangalore'), (26, 'Cust26', 31, 'Mumbai'), (27, 'Cust27', 25, 'Chennai'), (28, 'Cust28', 46, 'Bangalore'), (29, 'Cust29', 34, 'Chennai'), (30, 'Cust30', 35, 'Mumbai'), (31, 'Cust31', 20, 'Chennai'), (32, 'Cust32', 35, 'Bangalore'), (33, 'Cust33', 41, 'Mumbai'), (34, 'Cust34', 22, 'Bangalore'), (35, 'Cust35', 59, 'Hyderabad'), (36, 'Cust36', 32, 'Chennai'), (37, 'Cust37', 59, 'Hyderabad'), (38, 'Cust38', 35, 'Chennai'), (39, 'Cust39', 29, 'Mumbai'), (40, 'Cust40', 26, 'Bangalore'), (41, 'Cust41', 51, 'Chennai'), (42, 'Cust42', 25, 'Mumbai'), (43, 'Cust43', 52, 'Mumbai'), (44, 'Cust44', 44, 'Hyderabad'), (45, 'Cust45', 55, 'Mumbai'), (46, 'Cust46', 31, 'Hyderabad'), (47, 'Cust47', 24, 'Bangalore'), (48, 'Cust48', 60, 'Mumbai'), (49, 'Cust49', 26, 'Bangalore'), (50, 'Cust50', 58, 'Bangalore'), (51, 'Cust51', 60, 'Chennai'), (52, 'Cust52', 47, 'Chennai'), (53, 'Cust53', 48, 'Mumbai'), (54, 'Cust54', 25, 'Chennai'), (55, 'Cust55', 27, 'Mumbai'), (56, 'Cust56', 34, 'Chennai'), (57, 'Cust57', 54, 'Bangalore'), (58, 'Cust58', 29, 'Bangalore'), (59, 'Cust59', 51, 'Chennai'), (60, 'Cust60', 39, 'Hyderabad')], ["customer_id","name","age","city"])

policies = spark.createDataFrame([(101, 46, 'Life', 378, '2023-08-12'), (102, 60, 'Auto', 13450, '2023-09-10'), (103, 32, 'Life', -1484, '2023-06-29'), (104, 53, 'Auto', 16546, '2023-08-19'), (105, 50, 'Life', -4031, '2023-08-31'), (106, 53, 'Health', -3312, '2023-08-14'), (107, 33, 'Health', -4094, '2023-03-02'), (108, 58, 'Auto', 2618, '2023-10-05'), (109, 17, 'Life', 1509, '2023-02-26'), (110, 9, 'Life', 10514, '2023-05-10'), (111, 11, 'Health', 6363, '2023-02-20'), (112, 14, 'Life', 16596, '2023-10-23'), (113, 12, 'Auto', 9760, '2023-10-28'), (114, 23, 'Auto', -1102, '2023-10-08'), (115, 16, 'Auto', -3460, '2023-03-11'), (116, 15, 'Auto', 11693, '2023-08-18'), (117, 28, 'Life', 19215, '2023-02-09'), (118, 70, 'Auto', 1960, '2023-08-21'), (119, 18, 'Health', 15604, '2023-08-07'), (120, 69, 'Life', 2812, '2023-01-09'), (121, 34, 'Life', 10463, '2023-03-29'), (122, 65, 'Auto', 3456, '2023-05-07'), (123, 44, 'Health', 12480, '2023-09-26'), (124, 5, 'Life', -3927, '2023-05-12'), (125, 63, 'Health', 14463, '2023-02-17'), (126, 10, 'Health', -2972, '2023-06-27'), (127, 4, 'Health', 1795, '2023-07-08'), (128, 50, 'Auto', 17211, '2023-03-12'), (129, 52, 'Health', -993, '2023-06-15'), (130, 48, 'Health', 14425, '2023-05-09'), (131, 43, 'Health', 16770, '2023-05-16'), (132, 3, 'Health', -4360, '2023-07-28'), (133, 65, 'Life', -2649, '2023-09-07'), (134, 28, 'Auto', -733, '2023-10-08'), (135, 31, 'Life', 14704, '2023-04-09'), (136, 17, 'Health', 514, '2023-10-17'), (137, 31, 'Life', 11807, '2023-05-11'), (138, 59, 'Life', 8442, '2023-05-10'), (139, 37, 'Auto', -2390, '2023-05-15'), (140, 61, 'Auto', -830, '2023-04-14'), (141, 5, 'Auto', -1909, '2023-04-28'), (142, 45, 'Auto', 15333, '2023-10-14'), (143, 45, 'Auto', -4003, '2023-01-06'), (144, 69, 'Auto', -4821, '2023-10-13'), (145, 52, 'Health', 17575, '2023-07-19'), (146, 47, 'Auto', 16608, '2023-05-08'), (147, 25, 'Health', 11188, '2023-02-23'), (148, 67, 'Health', 8126, '2023-02-20'), (149, 18, 'Health', 17409, '2023-08-07'), (150, 8, 'Health', 9978, '2023-06-24'), (151, 12, 'Life', 15994, '2023-01-09'), (152, 16, 'Life', 11196, '2023-10-12'), (153, 54, 'Auto', 2962, '2023-01-29'), (154, 21, 'Auto', 4831, '2023-02-01'), (155, 19, 'Health', 11368, '2023-09-06'), (156, 17, 'Life', 11683, '2023-02-10'), (157, 19, 'Health', 5851, '2023-02-15'), (158, 30, 'Health', -4114, '2023-07-24'), (159, 65, 'Health', 14550, '2023-06-02'), (160, 28, 'Auto', 11985, '2023-08-20'), (161, 11, 'Health', 9652, '2023-02-20'), (162, 61, 'Health', 461, '2023-08-30'), (163, 2, 'Auto', -872, '2023-05-05'), (164, 6, 'Health', -3629, '2023-01-29'), (165, 45, 'Life', 3812, '2023-10-10'), (166, 4, 'Health', 14924, '2023-07-19'), (167, 21, 'Life', 1729, '2023-10-01'), (168, 56, 'Health', 13140, '2023-01-02'), (169, 33, 'Life', 15090, '2023-02-25'), (170, 1, 'Life', 7256, '2023-05-22'), (171, 18, 'Life', 6168, '2023-01-18'), (172, 56, 'Life', -4161, '2023-05-21'), (173, 41, 'Life', 19872, '2023-10-18'), (174, 1, 'Auto', 10181, '2023-03-12'), (175, 34, 'Life', 5894, '2023-08-04'), (176, 55, 'Life', -1322, '2023-02-22'), (177, 11, 'Auto', 5856, '2023-03-13'), (178, 35, 'Life', -605, '2023-07-03'), (179, 15, 'Auto', -624, '2023-01-19'), (180, 32, 'Life', 11973, '2023-06-13')], ["policy_id","customer_id","policy_type","premium","start_date"])

claims = spark.createDataFrame([(201, 128, 2016, '2023-07-04', 'Rejected'), (202, 189, 3524, '2023-06-10', 'Approved'), (203, 151, 6667, '2023-02-05', 'Rejected'), (204, 170, 5569, '2023-02-15', 'Rejected'), (205, 153, 13249, '2023-09-01', 'Rejected'), (206, 150, 8459, '2023-03-28', 'Rejected'), (207, 152, 11831, '2023-06-07', 'Pending'), (208, 150, 5932, '2023-05-22', 'Approved'), (209, 170, 8158, '2023-04-30', 'Pending'), (210, 156, 13267, '2023-07-13', 'Approved'), (211, 182, 12279, '2023-08-08', 'Rejected'), (212, 174, 13997, '2023-03-21', 'Pending'), (213, 119, 7369, '2023-08-23', 'Rejected'), (214, 168, 14982, '2023-06-08', 'Pending'), (215, 146, 13203, '2023-03-02', 'Rejected'), (216, 161, 4108, '2023-07-07', 'Approved'), (217, 194, -344, '2023-05-22', 'Pending'), (218, 125, 9804, '2023-09-13', 'Approved'), (219, 112, 6109, '2023-02-17', 'Pending'), (220, 162, 13753, '2023-07-16', 'Pending'), (221, 190, 11134, '2023-04-12', 'Approved'), (222, 143, 3817, '2023-10-11', 'Rejected'), (223, 171, 9525, '2023-01-08', 'Rejected'), (224, 156, 1400, '2023-03-17', 'Pending'), (225, 198, -1849, '2023-03-24', 'Approved'), (226, 181, 6380, '2023-08-07', 'Approved'), (227, 122, 14163, '2023-10-01', 'Rejected'), (228, 127, 13644, '2023-06-14', 'Approved'), (229, 154, 8419, '2023-03-17', 'Pending'), (230, 138, 12838, '2023-05-30', 'Approved'), (231, 127, 10296, '2023-04-22', 'Rejected'), (232, 134, 5878, '2023-08-24', 'Rejected'), (233, 145, 2140, '2023-05-28', 'Rejected'), (234, 114, 13117, '2023-03-25', 'Rejected'), (235, 141, 3192, '2023-06-11', 'Approved'), (236, 176, 5394, '2023-07-14', 'Approved'), (237, 155, 13537, '2023-04-09', 'Rejected'), (238, 108, 10908, '2023-09-29', 'Rejected'), (239, 135, 12294, '2023-09-27', 'Approved'), (240, 185, 1512, '2023-10-11', 'Pending'), (241, 171, 105, '2023-01-23', 'Approved'), (242, 188, 2507, '2023-08-06', 'Pending'), (243, 189, -425, '2023-09-22', 'Rejected'), (244, 119, -1334, '2023-06-25', 'Rejected'), (245, 142, -927, '2023-03-09', 'Pending'), (246, 126, 3636, '2023-09-07', 'Approved'), (247, 157, 1397, '2023-07-28', 'Rejected'), (248, 184, 11409, '2023-09-10', 'Rejected'), (249, 167, -1286, '2023-09-02', 'Pending'), (250, 141, -883, '2023-09-13', 'Pending'), (251, 118, 9669, '2023-05-26', 'Approved'), (252, 190, 7154, '2023-03-27', 'Pending'), (253, 157, 9716, '2023-04-05', 'Approved'), (254, 146, 13444, '2023-01-23', 'Approved'), (255, 112, 1432, '2023-10-08', 'Rejected'), (256, 106, 8508, '2023-04-03', 'Rejected'), (257, 103, 8969, '2023-10-11', 'Pending'), (258, 130, 2227, '2023-05-23', 'Pending'), (259, 125, 10332, '2023-10-08', 'Approved'), (260, 171, -1537, '2023-02-19', 'Approved'), (261, 180, 13362, '2023-02-13', 'Rejected'), (262, 157, 2359, '2023-04-24', 'Rejected'), (263, 154, 9574, '2023-08-07', 'Rejected'), (264, 163, 2059, '2023-04-03', 'Rejected'), (265, 101, 7373, '2023-03-14', 'Pending'), (266, 157, 9805, '2023-03-15', 'Approved'), (267, 180, 2053, '2023-07-07', 'Approved'), (268, 111, -964, '2023-07-30', 'Pending'), (269, 170, 2023, '2023-04-20', 'Pending'), (270, 109, 6252, '2023-10-25', 'Pending'), (271, 141, 10757, '2023-01-30', 'Approved'), (272, 164, 12245, '2023-07-14', 'Pending'), (273, 115, 12669, '2023-01-26', 'Rejected'), (274, 128, -731, '2023-08-18', 'Pending'), (275, 199, 9528, '2023-03-16', 'Pending'), (276, 140, 11854, '2023-07-24', 'Approved'), (277, 153, 10356, '2023-08-11', 'Pending'), (278, 192, 9657, '2023-08-24', 'Approved'), (279, 168, 13136, '2023-06-08', 'Approved'), (280, 161, 2476, '2023-06-27', 'Rejected')], ["claim_id","policy_id","claim_amount","claim_date","status"])

agents = spark.createDataFrame([(1, 'Agent1', 'South'), (2, 'Agent2', 'North'), (3, 'Agent3', 'West'), (4, 'Agent4', 'West'), (5, 'Agent5', 'South'), (6, 'Agent6', 'East'), (7, 'Agent7', 'South'), (8, 'Agent8', 'North'), (9, 'Agent9', 'East'), (10, 'Agent10', 'North'), (11, 'Agent11', 'West'), (12, 'Agent12', 'West'), (13, 'Agent13', 'South'), (14, 'Agent14', 'East'), (15, 'Agent15', 'North'), (16, 'Agent16', 'North'), (17, 'Agent17', 'South'), (18, 'Agent18', 'East'), (19, 'Agent19', 'East'), (20, 'Agent20', 'North')], ["agent_id","name","region"])

policy_agent = spark.createDataFrame([(125, 15), (105, 5), (115, 9), (130, 12), (143, 5), (120, 18), (143, 11), (116, 11), (166, 7), (108, 20), (131, 9), (180, 2), (106, 15), (102, 15), (161, 5), (110, 12), (145, 13), (158, 17), (132, 1), (148, 1), (134, 13), (136, 5), (174, 2), (171, 3), (114, 9), (129, 14), (109, 6), (141, 12), (166, 13), (142, 7), (133, 14), (163, 3), (156, 10), (145, 8), (119, 1), (121, 19), (123, 14), (111, 4), (122, 12), (134, 15), (101, 1), (135, 16), (133, 7), (123, 10), (144, 11), (124, 19), (178, 6), (165, 20), (158, 5), (115, 13), (102, 10), (106, 6), (150, 8), (126, 20), (140, 6), (136, 20), (153, 3), (178, 19), (124, 5), (160, 14), (151, 20), (162, 14), (131, 11), (109, 1), (124, 9), (109, 3), (106, 4), (143, 6), (175, 15), (165, 15), (116, 3), (130, 4), (174, 19), (164, 6), (148, 10), (177, 1), (105, 2), (114, 5), (130, 14), (156, 20)], ["policy_id","agent_id"])

policies = policies.withColumn("start_date", to_date(col("start_date")))
claims = claims.withColumn("claim_date", to_date(col("claim_date")))

print("All datasets loaded successfully")

All datasets loaded successfully


In [0]:
# Check schema and row counts for all tables

print("Customers:")
customers.printSchema()
print(f"Count: {customers.count()}\n")

print("Policies:")
policies.printSchema()
print(f"Count: {policies.count()}\n")

print("Claims:")
claims.printSchema()
print(f"Count: {claims.count()}\n")

print("Agents:")
agents.printSchema()
print(f"Count: {agents.count()}\n")

print("Policy_Agent:")
policy_agent.printSchema()
print(f"Count: {policy_agent.count()}")

In [0]:
# Check for null values in each table

print("Nulls in customers:")
customers.select([count(when(col(c).isNull(), c)).alias(c) for c in customers.columns]).show()

print("Nulls in policies:")
policies.select([count(when(col(c).isNull(), c)).alias(c) for c in policies.columns]).show()

print("Nulls in claims:")
claims.select([count(when(col(c).isNull(), c)).alias(c) for c in claims.columns]).show()

print("Nulls in agents:")
agents.select([count(when(col(c).isNull(), c)).alias(c) for c in agents.columns]).show()

print("Nulls in policy_agent:")
policy_agent.select([count(when(col(c).isNull(), c)).alias(c) for c in policy_agent.columns]).show()

In [0]:
# Identify negative premiums and claim amounts

print("Policies with negative premium:")
negative_premiums = policies.filter(col("premium") < 0)
print(f"Count: {negative_premiums.count()}")
negative_premiums.show(5)

print("Claims with negative claim_amount:")
negative_claims = claims.filter(col("claim_amount") < 0)
print(f"Count: {negative_claims.count()}")
negative_claims.show(5)

In [0]:
# Check for invalid foreign keys using left_anti join

print("Policies with invalid customer_id:")
invalid_policy_customers = policies.join(customers, "customer_id", "left_anti")
print(f"Count: {invalid_policy_customers.count()}")
invalid_policy_customers.show(5)

print("Claims with invalid policy_id:")
invalid_claim_policies = claims.join(policies, "policy_id", "left_anti")
print(f"Count: {invalid_claim_policies.count()}")
invalid_claim_policies.show(5)

print("Policy_agent with invalid policy_id:")
invalid_pa_policies = policy_agent.join(policies, "policy_id", "left_anti")
print(f"Count: {invalid_pa_policies.count()}")
invalid_pa_policies.show(5)

In [0]:
# Fill null values with appropriate defaults

customers = customers.fillna({"name": "Unknown", "city": "Unknown", "age": 0})
agents = agents.fillna({"name": "Unknown", "region": "Unknown"})

print("Null values handled successfully")

In [0]:
# Replace negative premiums and claim amounts with 0

policies = policies.withColumn("premium", when(col("premium") < 0, 0).otherwise(col("premium")))
claims = claims.withColumn("claim_amount", when(col("claim_amount") < 0, 0).otherwise(col("claim_amount")))

print(f"Fixed negative premiums. Policies count: {policies.count()}")
print(f"Fixed negative claims. Claims count: {claims.count()}")

In [0]:
# Trim whitespace from string columns

customers = customers.withColumn("name", trim(col("name"))).withColumn("city", trim(col("city")))
agents = agents.withColumn("name", trim(col("name"))).withColumn("region", trim(col("region")))
policies = policies.withColumn("policy_type", trim(col("policy_type")))
claims = claims.withColumn("status", trim(col("status")))

print("String columns trimmed successfully")

In [0]:
# Remove records with invalid foreign keys

policies_clean = policies.join(customers, "customer_id", "inner")
policies_clean = policies_clean.select(policies.columns)

claims_clean = claims.join(policies_clean, "policy_id", "inner")
claims_clean = claims_clean.select(claims.columns)

policy_agent_clean = policy_agent.join(policies_clean, "policy_id", "inner").join(agents, "agent_id", "inner")
policy_agent_clean = policy_agent_clean.select(policy_agent.columns)

print(f"Clean policies count: {policies_clean.count()}")
print(f"Clean claims count: {claims_clean.count()}")
print(f"Clean policy_agent count: {policy_agent_clean.count()}")

policies = policies_clean
claims = claims_clean
policy_agent = policy_agent_clean

In [0]:
# Create comprehensive validation report

total_customers = customers.count()
total_policies = policies.count()
total_claims = claims.count()
total_agents = agents.count()
total_policy_agent = policy_agent.count()

validation_report = spark.createDataFrame([
    ("total_customers", str(total_customers)),
    ("total_policies", str(total_policies)),
    ("total_claims", str(total_claims)),
    ("total_agents", str(total_agents)),
    ("total_policy_agent_mappings", str(total_policy_agent))
], ["metric", "count"])

print("Validation Report:")
validation_report.show(truncate=False)

In [0]:
# Calculate total premium by policy type

premium_by_type = policies.groupBy("policy_type").agg(
    sum("premium").alias("total_premium"),
    count("policy_id").alias("policy_count"),
    avg("premium").alias("avg_premium")
).orderBy(desc("total_premium"))

print("Premium Collection by Policy Type:")
premium_by_type.show()

Premium Collection by Policy Type:
+-----------+-------------+------------+-----------------+
|policy_type|total_premium|policy_count|      avg_premium|
+-----------+-------------+------------+-----------------+
|       Life|       204295|          26|           7857.5|
|     Health|       179036|          23|7784.173913043478|
|       Auto|       139034|          21|6620.666666666667|
+-----------+-------------+------------+-----------------+



In [0]:
# Analyze claims by status

claims_by_status = claims.groupBy("status").agg(
    count("claim_id").alias("claim_count"),
    sum("claim_amount").alias("total_claim_amount"),
    avg("claim_amount").alias("avg_claim_amount")
).orderBy(desc("total_claim_amount"))

print("Claims Analysis by Status:")
claims_by_status.show()

Claims Analysis by Status:
+--------+-----------+------------------+------------------+
|  status|claim_count|total_claim_amount|  avg_claim_amount|
+--------+-----------+------------------+------------------+
|Rejected|         25|            179586|           7183.44|
|Approved|         17|            133321| 7842.411764705882|
| Pending|         19|            114341|6017.9473684210525|
+--------+-----------+------------------+------------------+



In [0]:
# Calculate agent performance by region

df = policies.join(policy_agent, "policy_id").join(agents, "agent_id")

agent_performance = df.groupBy("region").agg(
    count("policy_id").alias("policies_sold"),
    sum("premium").alias("total_premium"),
    countDistinct("agent_id").alias("agent_count")
).orderBy(desc("total_premium"))

print("Agent Performance by Region:")
agent_performance.show()

Agent Performance by Region:
+------+-------------+-------------+-----------+
|region|policies_sold|total_premium|agent_count|
+------+-------------+-------------+-----------+
| North|           20|       159801|          6|
|  West|           14|        96522|          4|
| South|           19|        96269|          5|
|  East|           17|        77813|          4|
+------+-------------+-------------+-----------+



In [0]:
# Analyze customer risk based on premium vs claims

customer_risk = customers.join(policies, "customer_id", "left") \
    .join(claims, "policy_id", "left") \
    .groupBy("customer_id", "name", "city").agg(
        sum("premium").alias("total_premium"),
        sum("claim_amount").alias("total_claims")
    )

customer_risk = customer_risk.withColumn(
    "risk_ratio",
    when(col("total_premium") > 0, col("total_claims") / col("total_premium")).otherwise(0)
).orderBy(desc("risk_ratio"))

print("Top 10 High-Risk Customers:")
customer_risk.show(10)

Top 10 High-Risk Customers:
+-----------+------+---------+-------------+------------+------------------+
|customer_id|  name|     city|total_premium|total_claims|        risk_ratio|
+-----------+------+---------+-------------+------------+------------------+
|         46|Cust46|Hyderabad|          378|        7373|19.505291005291006|
|         58|Cust58|Bangalore|         2618|       10908| 4.166539343009931|
|         54|Cust54|  Chennai|         5924|       23605|3.9846387575962186|
|         16|Cust16|Bangalore|        11196|       24500| 2.188281529117542|
|         21|Cust21|  Chennai|        11391|       17993|1.5795803704679132|
|         59|Cust59|  Chennai|         8442|       12838|1.5207296849087895|
|          4| Cust4|Hyderabad|        18514|       23940|1.2930755104245435|
|         56|Cust56|  Chennai|        26280|       28118|1.0699391171993913|
|         19|Cust19|Hyderabad|        34772|       36814|1.0587254112504314|
|         32|Cust32|Bangalore|        23946|    

In [0]:
# Analyze monthly premium collection trends

monthly_trends = policies.withColumn("month", date_format(col("start_date"), "yyyy-MM")) \
    .groupBy("month").agg(
        count("policy_id").alias("policies_count"),
        sum("premium").alias("total_premium")
    ).orderBy("month")

print("Monthly Premium Collection Trends:")
monthly_trends.show()

Monthly Premium Collection Trends:
+-------+--------------+-------------+
|  month|policies_count|total_premium|
+-------+--------------+-------------+
|2023-01|             7|        38264|
|2023-02|            10|        85382|
|2023-03|             6|        43711|
|2023-04|             2|        14704|
|2023-05|            11|        85822|
|2023-06|             5|        21951|
|2023-07|             6|        34294|
|2023-08|             9|        79509|
|2023-09|             3|        37298|
|2023-10|            11|        81430|
+-------+--------------+-------------+



In [0]:
# Create temporary views for SQL analysis

customers.createOrReplaceTempView("customers_view")
policies.createOrReplaceTempView("policies_view")
claims.createOrReplaceTempView("claims_view")
agents.createOrReplaceTempView("agents_view")
policy_agent.createOrReplaceTempView("policy_agent_view")

print("Temporary views created successfully")

Temporary views created successfully


In [0]:
# Find top 3 risky customers per city using CTE and window functions

top_risky_sql = """
WITH customer_premium AS (
    SELECT 
        c.customer_id,
        c.name,
        c.city,
        SUM(p.premium) AS total_premium
    FROM customers_view c
    LEFT JOIN policies_view p ON c.customer_id = p.customer_id
    GROUP BY c.customer_id, c.name, c.city
),
customer_claims AS (
    SELECT 
        p.customer_id,
        SUM(cl.claim_amount) AS total_claims
    FROM policies_view p
    LEFT JOIN claims_view cl ON p.policy_id = cl.policy_id
    GROUP BY p.customer_id
),
customer_risk AS (
    SELECT 
        cp.customer_id,
        cp.name,
        cp.city,
        COALESCE(cp.total_premium, 0) AS total_premium,
        COALESCE(cc.total_claims, 0) AS total_claims,
        CASE 
            WHEN COALESCE(cp.total_premium, 0) > 0 
            THEN COALESCE(cc.total_claims, 0) / cp.total_premium 
            ELSE 0 
        END AS risk_ratio
    FROM customer_premium cp
    LEFT JOIN customer_claims cc ON cp.customer_id = cc.customer_id
),
ranked_risk AS (
    SELECT 
        customer_id,
        name,
        city,
        total_premium,
        total_claims,
        risk_ratio,
        ROW_NUMBER() OVER (PARTITION BY city ORDER BY risk_ratio DESC) AS rank
    FROM customer_risk
)
SELECT 
    customer_id,
    name,
    city,
    total_premium,
    total_claims,
    ROUND(risk_ratio, 4) AS risk_ratio,
    rank
FROM ranked_risk
WHERE rank <= 3
ORDER BY city, rank
"""

top_risky_customers = spark.sql(top_risky_sql)
print("Top 3 Risky Customers per City:")
top_risky_customers.show(20, truncate=False)

Top 3 Risky Customers per City:
+-----------+------+---------+-------------+------------+----------+----+
|customer_id|name  |city     |total_premium|total_claims|risk_ratio|rank|
+-----------+------+---------+-------------+------------+----------+----+
|58         |Cust58|Bangalore|2618         |10908       |4.1665    |1   |
|16         |Cust16|Bangalore|11196        |24500       |2.1883    |2   |
|32         |Cust32|Bangalore|11973        |24384       |2.0366    |3   |
|54         |Cust54|Chennai  |2962         |23605       |7.9693    |1   |
|21         |Cust21|Chennai  |6560         |17993       |2.7428    |2   |
|56         |Cust56|Chennai  |13140        |28118       |2.1399    |3   |
|46         |Cust46|Hyderabad|378          |7373        |19.5053   |1   |
|19         |Cust19|Hyderabad|17219        |36814       |2.138     |2   |
|4          |Cust4 |Hyderabad|16719        |23940       |1.4319    |3   |
|53         |Cust53|Mumbai   |16546        |8508        |0.5142    |1   |
|14   

In [0]:
# Analyze monthly claim trends to find customers with increasing claims

monthly_claims_sql = """
WITH monthly_claims AS (
    SELECT 
        p.customer_id,
        c.name,
        DATE_FORMAT(cl.claim_date, 'yyyy-MM') AS claim_month,
        SUM(cl.claim_amount) AS monthly_claim_amount,
        COUNT(cl.claim_id) AS claim_count
    FROM policies_view p
    JOIN claims_view cl ON p.policy_id = cl.policy_id
    JOIN customers_view c ON p.customer_id = c.customer_id
    GROUP BY p.customer_id, c.name, DATE_FORMAT(cl.claim_date, 'yyyy-MM')
),
claim_trends AS (
    SELECT 
        customer_id,
        name,
        claim_month,
        monthly_claim_amount,
        claim_count,
        LAG(monthly_claim_amount) OVER (PARTITION BY customer_id ORDER BY claim_month) AS prev_month_claims
    FROM monthly_claims
)
SELECT 
    customer_id,
    name,
    claim_month,
    monthly_claim_amount,
    claim_count,
    prev_month_claims,
    CASE 
        WHEN prev_month_claims IS NOT NULL AND monthly_claim_amount > prev_month_claims 
        THEN 'Increasing' 
        WHEN prev_month_claims IS NOT NULL AND monthly_claim_amount < prev_month_claims 
        THEN 'Decreasing'
        ELSE 'Stable'
    END AS trend
FROM claim_trends
WHERE prev_month_claims IS NOT NULL
ORDER BY customer_id, claim_month
"""

monthly_claim_trends = spark.sql(monthly_claims_sql)
print("Monthly Claim Trends:")
monthly_claim_trends.show(20, truncate=False)

Monthly Claim Trends:
+-----------+------+-----------+--------------------+-----------+-----------------+----------+
|customer_id|name  |claim_month|monthly_claim_amount|claim_count|prev_month_claims|trend     |
+-----------+------+-----------+--------------------+-----------+-----------------+----------+
|1          |Cust1 |2023-03    |13997               |1          |5569             |Increasing|
|1          |Cust1 |2023-04    |10181               |2          |13997            |Decreasing|
|4          |Cust4 |2023-06    |13644               |1          |10296            |Increasing|
|5          |Cust5 |2023-06    |3192                |1          |10757            |Decreasing|
|5          |Cust5 |2023-09    |0                   |1          |3192             |Decreasing|
|8          |Cust8 |2023-05    |5932                |1          |8459             |Decreasing|
|11         |Cust11|2023-07    |4108                |2          |2476             |Increasing|
|14         |Cust14|2023-10 

In [0]:
# Analyze policy concentration by type and city

policy_concentration_sql = """
WITH policy_by_city_type AS (
    SELECT 
        c.city,
        p.policy_type,
        COUNT(p.policy_id) AS policy_count,
        SUM(p.premium) AS total_premium,
        AVG(p.premium) AS avg_premium
    FROM customers_view c
    JOIN policies_view p ON c.customer_id = p.customer_id
    GROUP BY c.city, p.policy_type
),
city_totals AS (
    SELECT 
        city,
        SUM(policy_count) AS total_city_policies,
        SUM(total_premium) AS total_city_premium
    FROM policy_by_city_type
    GROUP BY city
)
SELECT 
    pct.city,
    pct.policy_type,
    pct.policy_count,
    pct.total_premium,
    ROUND(pct.avg_premium, 2) AS avg_premium,
    ROUND(100.0 * pct.policy_count / ct.total_city_policies, 2) AS policy_percentage,
    ROUND(100.0 * pct.total_premium / ct.total_city_premium, 2) AS premium_percentage
FROM policy_by_city_type pct
JOIN city_totals ct ON pct.city = ct.city
ORDER BY pct.city, pct.total_premium DESC
"""

policy_concentration = spark.sql(policy_concentration_sql)
print("Policy Concentration by City and Type:")
policy_concentration.show(30, truncate=False)

Policy Concentration by City and Type:
+---------+-----------+------------+-------------+-----------+-----------------+------------------+
|city     |policy_type|policy_count|total_premium|avg_premium|policy_percentage|premium_percentage|
+---------+-----------+------------+-------------+-----------+-----------------+------------------+
|Bangalore|Life       |9           |81991        |9110.11    |47.37            |50.76             |
|Bangalore|Auto       |8           |68363        |8545.38    |42.11            |42.32             |
|Bangalore|Health     |2           |11188        |5594.0     |10.53            |6.93              |
|Chennai  |Life       |8           |69746        |8718.25    |50.00            |58.74             |
|Chennai  |Health     |5           |41207        |8241.4     |31.25            |34.70             |
|Chennai  |Auto       |3           |7793         |2597.67    |18.75            |6.56              |
|Hyderabad|Health     |7           |62433        |8919.0     

In [0]:
# Use window functions to rank agents by total premium

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

agent_premium = policies.join(policy_agent, "policy_id") \
    .join(agents, "agent_id") \
    .groupBy("agent_id", "name", "region").agg(
        sum("premium").alias("total_premium"),
        count("policy_id").alias("policies_sold")
    )

# Overall ranking
window_overall = Window.orderBy(desc("total_premium"))
agent_premium = agent_premium.withColumn("overall_rank", row_number().over(window_overall))

# Regional ranking
window_region = Window.partitionBy("region").orderBy(desc("total_premium"))
agent_premium = agent_premium.withColumn("region_rank", rank().over(window_region))

print("Agent Rankings by Premium:")
agent_premium.orderBy("overall_rank").show(20, truncate=False)

Agent Rankings by Premium:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-------+------+-------------+-------------+------------+-----------+
|agent_id|name   |region|total_premium|policies_sold|overall_rank|region_rank|
+--------+-------+------+-------------+-------------+------------+-----------+
|14      |Agent14|East  |38890        |4            |1           |1          |
|10      |Agent10|North |37613        |3            |2           |1          |
|20      |Agent20|North |34621        |6            |3           |2          |
|13      |Agent13|South |32499        |4            |4           |1          |
|7       |Agent7 |South |30257        |2            |5           |2          |
|11      |Agent11|West  |28463        |3            |6           |1          |
|8       |Agent8 |North |27553        |2            |7           |3          |
|12      |Agent12|West  |24939        |3            |8           |2          |
|1       |Agent1 |South |23347        |5            |9           |3          |
|15      |Agent15|North |23156        |5            

In [0]:
# Rank customers by risk score and calculate percentiles

from pyspark.sql.functions import percent_rank, ntile

# Use aliases to avoid ambiguous column references
customers_alias = customers.alias("c")
policies_alias = policies.alias("p")
claims_alias = claims.alias("cl")

customer_metrics = customers_alias.join(policies_alias, "customer_id", "left") \
    .join(claims_alias, col("p.policy_id") == col("cl.policy_id"), "left") \
    .groupBy(col("c.customer_id"), col("c.name"), col("c.city")).agg(
        sum("p.premium").alias("total_premium"),
        sum("cl.claim_amount").alias("total_claims"),
        count(col("p.policy_id")).alias("policy_count")
    )

customer_metrics = customer_metrics.withColumn(
    "risk_score",
    when(col("total_premium") > 0, col("total_claims") / col("total_premium")).otherwise(0)
)

# Add ranking columns
window_risk = Window.orderBy(desc("risk_score"))
window_city = Window.partitionBy("city").orderBy(desc("risk_score"))

customer_metrics = customer_metrics \
    .withColumn("overall_rank", row_number().over(window_risk)) \
    .withColumn("city_rank", rank().over(window_city)) \
    .withColumn("risk_percentile", percent_rank().over(window_risk)) \
    .withColumn("risk_quartile", ntile(4).over(window_risk))

print("Customer Risk Rankings (Top 15):")
customer_metrics.orderBy("overall_rank").show(15, truncate=False)

Customer Risk Rankings (Top 15):


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------+---------+-------------+------------+------------+------------------+------------+---------+-------------------+-------------+
|customer_id|name  |city     |total_premium|total_claims|policy_count|risk_score        |overall_rank|city_rank|risk_percentile    |risk_quartile|
+-----------+------+---------+-------------+------------+------------+------------------+------------+---------+-------------------+-------------+
|46         |Cust46|Hyderabad|378          |7373        |1           |19.505291005291006|1           |1        |0.0                |1            |
|58         |Cust58|Bangalore|2618         |10908       |1           |4.166539343009931 |2           |1        |0.01694915254237288|1            |
|54         |Cust54|Chennai  |5924         |23605       |2           |3.9846387575962186|3           |1        |0.03389830508474576|1            |
|16         |Cust16|Bangalore|11196        |24500       |2           |2.188281529117542 |4           |2        |0.0508

In [0]:
# Find top 2 performing agents in each region

top_agents_per_region = policies.join(policy_agent, "policy_id") \
    .join(agents, "agent_id") \
    .groupBy("region", "agent_id", agents["name"]).agg(
        sum("premium").alias("total_premium"),
        count("policy_id").alias("policies_sold"),
        avg("premium").alias("avg_premium")
    )

window_top_agents = Window.partitionBy("region").orderBy(desc("total_premium"))
top_agents_per_region = top_agents_per_region \
    .withColumn("rank", row_number().over(window_top_agents)) \
    .filter(col("rank") <= 2)

print("Top 2 Agents per Region:")
top_agents_per_region.orderBy("region", "rank").show(10, truncate=False)

Top 2 Agents per Region:
+------+--------+-------+-------------+-------------+------------------+----+
|region|agent_id|name   |total_premium|policies_sold|avg_premium       |rank|
+------+--------+-------+-------------+-------------+------------------+----+
|East  |14      |Agent14|38890        |4            |9722.5            |1   |
|East  |19      |Agent19|20644        |4            |5161.0            |2   |
|North |10      |Agent10|37613        |3            |12537.666666666666|1   |
|North |20      |Agent20|34621        |6            |5770.166666666667 |2   |
|South |13      |Agent13|32499        |4            |8124.75           |1   |
|South |7       |Agent7 |30257        |2            |15128.5           |2   |
|West  |11      |Agent11|28463        |3            |9487.666666666666 |1   |
|West  |12      |Agent12|24939        |3            |8313.0            |2   |
+------+--------+-------+-------------+-------------+------------------+----+



In [0]:
# Calculate running totals and cumulative metrics by month

from pyspark.sql.functions import month, year

monthly_data = policies.withColumn("year_month", date_format(col("start_date"), "yyyy-MM")) \
    .groupBy("year_month").agg(
        count("policy_id").alias("policies_sold"),
        sum("premium").alias("monthly_premium")
    ).orderBy("year_month")

# Calculate running totals
window_cumulative = Window.orderBy("year_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

monthly_data = monthly_data \
    .withColumn("cumulative_policies", sum("policies_sold").over(window_cumulative)) \
    .withColumn("cumulative_premium", sum("monthly_premium").over(window_cumulative)) \
    .withColumn("3_month_avg_premium", avg("monthly_premium").over(Window.orderBy("year_month").rowsBetween(-2, 0)))

print("Monthly Cumulative Analysis:")
monthly_data.show(15, truncate=False)

Monthly Cumulative Analysis:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-------------+---------------+-------------------+------------------+-------------------+
|year_month|policies_sold|monthly_premium|cumulative_policies|cumulative_premium|3_month_avg_premium|
+----------+-------------+---------------+-------------------+------------------+-------------------+
|2023-01   |7            |38264          |7                  |38264             |38264.0            |
|2023-02   |10           |85382          |17                 |123646            |61823.0            |
|2023-03   |6            |43711          |23                 |167357            |55785.666666666664 |
|2023-04   |2            |14704          |25                 |182061            |47932.333333333336 |
|2023-05   |11           |85822          |36                 |267883            |48079.0            |
|2023-06   |5            |21951          |41                 |289834            |40825.666666666664 |
|2023-07   |6            |34294          |47                 |324128            |4